# 🩺 Reprocesamiento de P6_MULTIVARIADO (Kaggle GPU)
### Regeneración Sincronizada de Signos Vitales Hemodinámicos (Presión Sistólica y Diastólica)

Este cuaderno permite regenerar y evaluar el perfil **P6_MULTIVARIADO** para todos los escenarios ($E_1$ a $E_9$) utilizando las **GPUs de Kaggle (Tesla T4 o P100)**:
* **Entrada requerida:** Dataset adjunto en Kaggle con la carpeta `resultados` (que contiene las subcarpetas `E1_resultados` a `E9_resultados` con sus respectivos adaptadores `student_adapter`).
* **Salida generada:** Series sintéticas multivariadas de 2 canales, re-entrenamiento de la RNN evaluadora (LSTM), cálculo de métricas macro y desglosadas por variable, y gráficos limpios de 2 paneles (`grafica_prediccion.png`).
* **Descarga final:** Un archivo `resultados_p6_actualizados.zip` listo para descargar desde el panel de salida de Kaggle.


## 1. Instalación de Dependencias y Preparación del Entorno


In [ ]:
# Desinstalar torchao incompatible preinstalado en Kaggle (0.10.0 vs >=0.16.0 requerido por peft)
!pip uninstall -y torchao
!pip install -q -U "transformers>=4.45.0" "peft>=0.13.0" accelerate bitsandbytes datasets matplotlib evaluate

import os
import gc
import sys
import glob
import shutil
import json
from pathlib import Path
import torch

print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Dispositivo GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM Total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


## 2. Clonar Repositorio del Proyecto (Rama `kaggle_experimentos`)


In [ ]:
REPO_DIR = Path("/kaggle/working/proyecto_tesis")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

!git clone -b kaggle_experimentos https://github.com/danyr59/tesis_antes.git /kaggle/working/proyecto_tesis
%cd /kaggle/working/proyecto_tesis


## 3. Autenticación de Hugging Face (para LLaMA 3.2 y Gemma 2)
> Agrega tu token en Kaggle: **Add-ons -> Secrets -> HF_TOKEN**, o pégalo directamente en la celda abajo.


In [ ]:
import os

# Intentar cargar desde Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = hf_token
    print("✅ HF_TOKEN cargado desde Kaggle Secrets.")
except Exception:
    token_manual = ""  # Ej: "hf_..."
    if token_manual:
        os.environ["HF_TOKEN"] = token_manual
        print("✅ HF_TOKEN configurado manualmente.")
    else:
        print("ℹ️ No se detectó HF_TOKEN. Si algún modelo lo requiere, configúralo en Kaggle Secrets.")

token_activo = os.environ.get("HF_TOKEN")
if token_activo:
    try:
        from huggingface_hub import login
        login(token=token_activo)
        print("✅ Sesión iniciada en huggingface_hub.")
    except Exception as e:
        print(f"⚠️ Nota login: {e}")


## 4. Localizar y Preparar la Carpeta `resultados`
El script busca automáticamente el dataset adjunto en `/kaggle/input` y lo copia a `/kaggle/working/resultados` para habilitar escritura.


In [ ]:
import glob
import shutil
from pathlib import Path

# Buscar la carpeta de resultados en /kaggle/input
input_candidates = glob.glob("/kaggle/input/**/E1_resultados", recursive=True) + glob.glob("/kaggle/input/**/E2_resultados", recursive=True)
resultados_working = Path("/kaggle/working/resultados")

if input_candidates:
    src_dir = Path(input_candidates[0]).parent
    print(f"📂 Dataset encontrado en: {src_dir}")
    if resultados_working.exists():
        shutil.rmtree(resultados_working)
    print("Copiando resultados a /kaggle/working/resultados (esto toma unos segundos)...")
    shutil.copytree(src_dir, resultados_working)
    print("✅ Resultados preparados en /kaggle/working/resultados")
else:
    # Si se subió como archivo ZIP
    zips = glob.glob("/kaggle/input/**/*.zip", recursive=True)
    if zips:
        print(f"📦 Descomprimiendo archivo {zips[0]}...")
        import zipfile
        with zipfile.ZipFile(zips[0], 'r') as zip_ref:
            zip_ref.extractall("/kaggle/working/")
        print("✅ Descomprimido exitosamente.")
    else:
        print("⚠️ No se encontró la carpeta de resultados en /kaggle/input.")
        print("Por favor, asegúrate de adjuntar tu dataset 'resultados' usando el botón '+ Add Input' a la derecha.")


## 5. Ejecutar Reprocesamiento de P6_MULTIVARIADO
Puedes seleccionar los escenarios a procesar. Por defecto procesa los 9 escenarios (`E1,E2,E3,E4,E5,E6,E7,E8,E9`).


In [ ]:
# Configura los escenarios que deseas reprocesar:
# Ejemplos: "E1,E2,E3,E4,E5,E6,E7,E8,E9" (todos) o "E5,E6,E7,E8,E9" (solo los restantes)
ESCENARIOS_A_PROCESAR = "E1,E2,E3,E4,E5,E6,E7,E8,E9"

!python scripts/reprocesar_p6.py \
    --resultados_dir /kaggle/working/resultados \
    --escenarios {ESCENARIOS_A_PROCESAR}


## 6. Inspección de Gráficas Multivariadas Generadas


In [ ]:
from IPython.display import Image, display

print("=" * 80)
print("📊 GRÁFICAS P6_MULTIVARIADO ACTUALIZADAS")
print("=" * 80)

for esc_id in ["E1", "E2", "E3", "E4", "E5", "E6", "E7", "E8", "E9"]:
    plot_file = Path(f"/kaggle/working/resultados/{esc_id}_resultados/P6_MULTIVARIADO/grafica_prediccion.png")
    metricas_file = Path(f"/kaggle/working/resultados/{esc_id}_resultados/P6_MULTIVARIADO/metricas.json")
    if plot_file.exists():
        r2_info = "N/A"
        if metricas_file.exists():
            with open(metricas_file) as f:
                m_data = json.load(f)
                r2_info = m_data.get("downstream", {}).get("r2", "N/A")
        print(f"\n🎯 Escenario {esc_id} (P6_MULTIVARIADO) — R² Macro: {r2_info}")
        display(Image(filename=str(plot_file), width=850))


## 7. Empaquetar Resultados Actualizados para Descarga


In [ ]:
# Comprimir la carpeta de resultados actualizada
output_zip = "/kaggle/working/resultados_p6_actualizados.zip"
if Path(output_zip).exists():
    os.remove(output_zip)

print("📦 Comprimiendo resultados actualizados...")
!cd /kaggle/working && zip -r -q resultados_p6_actualizados.zip resultados/
print("🎉 ¡Proceso completo!")
print("Puedes descargar 'resultados_p6_actualizados.zip' desde el panel derecho de Kaggle (Sección Output).")
